# Image verification lab ? Colab pixel-analysis baseline

This notebook runs in a **Google-hosted Python runtime**, avoiding the Windows runtime restriction on your laptop without changing its security settings. It is a notebook, not the local Streamlit website.

**Before running:** uploads go to Google's runtime. Only upload images you are comfortable processing there. Uploaded files and displayed images/results may persist in the runtime or saved notebook output; clear outputs before sharing. This notebook does not mount Google Drive or send images to Hugging Face. Hugging Face is contacted to download model files.

Use a new hosted CPU runtime (GPU is not needed). Run cells from top to bottom with each cell's play button. Wait for setup and model loading to finish. If rerunning setup after imports, restart the session and rerun from the top.

Two separate questions:
1. Does the image have camera-photo-like or AI-like pixel patterns?
2. Is there evidence of local editing? **Not implemented; inconclusive.**

The model cannot establish that an image or depicted event is legitimate. A camera photo can also be edited.


## 1. Install the CPU runtime dependencies
This changes packages only in the temporary Colab runtime. Downloads can take several minutes. The pinned CPU stack does not require torchvision; removing it avoids optional binary-extension conflicts.


In [ ]:
import subprocess
import sys

if 'google.colab' not in sys.modules:
    import importlib.util
    if importlib.util.find_spec('google.colab') is None:
        raise RuntimeError('Open this notebook in Google Colab using a hosted runtime.')

subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch==2.10.0+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.6', 'Pillow>=11,<13'])
print('Setup complete. Continue to cell 2.')


## 2. Load the model
Baseline: [Organika/sdxl-detector](https://huggingface.co/Organika/sdxl-detector), pinned revision `b37fede`. Attribution: Organika / Colby / umm-maybe, as described in its model card. Listed license: CC-BY-NC-3.0, for educational/noncommercial use; review terms before other uses.

The model was fine-tuned on Wikimedia/SDXL pairs. Other generators, illustrations, screenshots, compression, and resizing can lead to wrong results. Its published metrics are not measured accuracy on your images. Weights are downloaded into this runtime; expect several hundred MB.


In [ ]:
"""Experimental pixel-only SDXL baseline; no EXIF fields enter the model."""

import argparse
import math
from pathlib import Path

from PIL import Image

MODEL_ID = "Organika/sdxl-detector"
MODEL_REVISION = "b37fede"
CACHE_DIR = Path.cwd() / ".models"


def load_detector(download=False):
    # Check this explicitly: torch._dynamo can swallow its DLL ImportError and
    # replace it with an unrelated-looking NP_SUPPORTED_MODULES import failure.
    import numpy.fft  # noqa: F401
    import torch
    from transformers import AutoImageProcessor, AutoModelForImageClassification

    options = dict(
        revision=MODEL_REVISION,
        cache_dir=str(CACHE_DIR),
        local_files_only=not download,
        trust_remote_code=False,
    )
    processor = AutoImageProcessor.from_pretrained(MODEL_ID, use_fast=False, **options)
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_ID, use_safetensors=True, **options
    )
    labels = {int(key): value.lower() for key, value in model.config.id2label.items()}
    if set(labels.values()) != {"artificial", "human"}:
        raise ValueError("Unexpected model labels; inference stopped.")
    model.to("cpu").eval()
    return processor, model, torch, labels


def interpret_score(score):
    if not math.isfinite(score) or not 0 <= score <= 1:
        raise ValueError("Invalid model score.")
    # Exploratory bands only, not thresholds validated on the user's data.
    if score >= 0.9:
        return "AI-like signal (experimental)"
    if score <= 0.1:
        return "Camera-photo-like signal (experimental)"
    return "Inconclusive"


def analyze_pixels(image, detector):
    processor, model, torch, labels = detector
    # Make a new RGB image so EXIF and other metadata cannot enter inference.
    rgb = image.convert("RGB")
    pixels = Image.frombytes("RGB", rgb.size, rgb.tobytes())
    inputs = processor(images=pixels, return_tensors="pt")
    with torch.inference_mode():
        probabilities = model(**inputs).logits.softmax(dim=-1)[0]
    ai_index = next(index for index, name in labels.items() if name == "artificial")
    score = float(probabilities[ai_index].item())
    return {"ai_score": score, "signal": interpret_score(score)}



# This is a standalone copy of the project's detector functions.
# The actual model architecture comes from Transformers, not custom remote code.
detector = load_detector(download=True)
import torch
import transformers
import PIL
print('Model ready:', MODEL_ID, '@', MODEL_REVISION)
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__, '| transformers:', transformers.__version__, '| Pillow:', PIL.__version__)


## 3. Upload and analyze JPEG/PNG images
Run this cell and select one or more files. Upload limits are checked after browser transfer, so choose files no larger than 20 MB. For another set of photos, rerun this cell.

A score near 1 favors the model's artificial class; near 0 favors its human class. This is **not a calibrated probability of authenticity**. The 0.10/0.90 bands are exploratory and have not been validated on your dataset. Overall authenticity stays inconclusive.


In [ ]:
import io
import warnings
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps, UnidentifiedImageError

results = []
uploaded = files.upload()
if not uploaded:
    print('No images selected. Run this cell again to upload.')

for filename, data in uploaded.items():
    print('\nFile:', filename)
    try:
        if len(data) > 20 * 1024 * 1024:
            raise ValueError('Choose an image no larger than 20 MB.')
        with warnings.catch_warnings():
            warnings.simplefilter('error', Image.DecompressionBombWarning)
            with Image.open(io.BytesIO(data)) as original:
                if original.format not in {'JPEG', 'PNG'}:
                    raise ValueError('Only JPEG and PNG contents are supported.')
                original.load()
                image_format, dimensions = original.format, original.size
                try:
                    image = ImageOps.exif_transpose(original).convert('RGB')
                except Exception:
                    image = original.convert('RGB')
        preview = image.copy()
        preview.thumbnail((1000, 1000))
        display(preview)
        print('Format:', image_format, '| Stored dimensions:', dimensions)
        result = analyze_pixels(image, detector)
        print('1. Generation model indication:', result['signal'])
        print('AI-class model score:', format(result['ai_score'], '.3f'))
        print('Authenticity conclusion: inconclusive ? baseline not validated on your target data.')
        print('2. Local editing: inconclusive ? edit localization is not implemented.')
        results.append({'filename': filename, **result,
                        'authenticity': 'inconclusive', 'local_editing': 'not implemented / inconclusive',
                        'model': MODEL_ID, 'revision': MODEL_REVISION})
    except (ValueError, OSError, SyntaxError, Image.DecompressionBombWarning) as error:
        print('Cannot analyze this image. Result: inconclusive.', str(error))
    except Exception as error:
        print('Analysis failed. Result: inconclusive.', type(error).__name__, str(error))

print('\nFinished. No detection score is a proof of authenticity.')


## 4. Optionally download a results report
This report records model scores and versions, not proven authenticity labels. Filenames are included. Only the latest upload batch is exported. Clear notebook outputs before sharing it if they contain private images or filenames.


In [ ]:
import json
from pathlib import Path

if not globals().get('results'):
    print('No successful results to export. Run the upload cell first.')
else:
    report = {
        'model': MODEL_ID, 'revision': MODEL_REVISION,
        'versions': {'python': sys.version.split()[0], 'torch': torch.__version__,
                     'transformers': transformers.__version__, 'pillow': PIL.__version__},
        'exploratory_bands': {'camera_photo_like_max': 0.10, 'ai_like_min': 0.90},
        'validated_on_target_data': False,
        'results': results,
    }
    Path('analysis_results.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    files.download('analysis_results.json')


## Next: scientific evaluation
A successful run checks functionality, not accuracy. Collect labeled camera photos and AI-generated images from multiple generators. Keep originals, edits, recompressed copies, and near duplicates in the same split to prevent leakage. Select thresholds on validation data, then freeze them before evaluating an untouched test set, including unseen generators.

Report precision, recall, F1, camera-photo false-positive rate, and results per source/generator. For inconclusive outcomes, report the inconclusive rate and coverage as well. Check calibration before calling scores probabilities. Keep local edit localization (TruFor) and signed history checks (C2PA) separate.

When finished, disconnect and delete the runtime using Colab's Runtime menu. Downloads and uploaded files may need to be supplied again in a new session.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Colab file uploads](https://colab.research.google.com/notebooks/io.ipynb), [model card](https://huggingface.co/Organika/sdxl-detector), [TruFor](https://github.com/grip-unina/TruFor).
